In [12]:
from CNN_NAS.ChildCNNModel import ChildCNNModel
# Some magic so that the notebook will reload the external python script file any time you edit and save the .py file;
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os

import utils

import logging
logging.basicConfig(level=logging.INFO, filename=os.path.join(os.getcwd(), 'log.log'), filemode='w')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

device = utils.get_device_available()
print(torch.__version__)
print(device)

2.4.1
cuda


In [14]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

## CIFAR

In [15]:
dataset="cifar"

data_path = utils.check_cifar_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + 'cifar/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")



Height: 32
Width: 32
Number of channels: 3
Number of classes:  10


## Load  predefined model encoding

In [16]:
import predefined_models
# Defined by data

base_model_encoding_dict = {}

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Alexnet"] = predefined_models.get_alexnet(input_channels=num_channels, output_dim=num_classes)
# 

# base_model = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)

# print(base_model)

In [17]:
for base_model in base_model_encoding_dict:
    

SyntaxError: unexpected EOF while parsing (1155890307.py, line 2)

In [18]:
from CNN_NAS.CNNController import CNNController

#Load and run
logger.info("###############################################")
logger.info("STARTING TRAINING")
logger.info("###############################################")


# print(model)

for base_model in base_model_encoding_dict:
    total_epochs=10
    name = base_model
    for i in range(3):
        model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
        loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label)
        print(f"Model {name} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10

Model Benchmark_Model Test Accuracy: (0.7249, 0.809419242143631) with total epochs 10 in dataset cifar for time 31.670965909957886
Model Benchmark_Model Test Accuracy: (0.7345, 1.1888421738147736) with total epochs 20 in dataset cifar for time 62.18864583969116
Model Benchmark_Model Test Accuracy: (0.7183, 1.8332150459289551) with total epochs 30 in dataset cifar for time 93.94693088531494
